In [1]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np


## Define target
We're trying to predict if a user will posts in the second week. 

In [2]:
posts_path = "../data/posting/filtered/chunk_0_posts.parquet"
posts_table = pq.read_table(posts_path)
posts_df = posts_table.to_pandas()

join_dates_path = "../data/posting/processed/join_dates.parquet"
join_dates_df = pd.read_parquet(join_dates_path)
if 'created_at' in join_dates_df.columns:
    join_dates_df = join_dates_df.rename(columns={'created_at': 'join_date'})

target_table = pd.DataFrame({'did_id': posts_df['did_id'].unique()})
target_table = target_table.merge(join_dates_df[['did_id', 'join_date']], on='did_id', how='left')

merged = posts_df.merge(target_table[['did_id', 'join_date']], on='did_id', how='left')

merged['days_since_join'] = ((merged['created_at'] - merged['join_date']).dt.total_seconds() / (24 * 3600)).round().astype('Int64')
second_week_posts = merged[(merged['days_since_join'].notna()) & (merged['days_since_join'] >= 7) & (merged['days_since_join'] <= 13)].copy()

second_week_counts = second_week_posts.groupby('did_id').size().reset_index(name='posts_week2')
target_table = target_table.merge(second_week_counts[['did_id', 'posts_week2']], on='did_id', how='left')
target_table['posts_week2'] = target_table['posts_week2'].fillna(0).astype(int)

print(target_table[['did_id', 'posts_week2']].head())

     did_id  posts_week2
0  13255698            2
1  27401584            0
2  12123752            1
3   9037590           39
4  32928599            0


# Feature Engineering

In [3]:
# Load user activity and coerce data type.
user_path = "../data/posting/processed/user_activity.parquet"
user_df = pd.read_parquet(user_path)
print('Loaded user activity rows:', len(user_df))

def ensure_vec7(x):
    try:
        if pd.isna(x):
            return [0]*7
    except Exception:
        pass
    if isinstance(x, list):
        if len(x) == 7:
            return [int(v) for v in x]
        # pad or trim
        lst = [int(v) for v in x[:7]] + [0]*max(0, 7 - len(x))
        return lst
    # try converting numpy/other sequence
    try:
        lst = list(x)
        lst = [int(v) for v in lst[:7]] + [0]*max(0, 7 - len(lst))
        return lst
    except Exception:
        return [0]*7

vec_cols = ['posts_vec','blocks_actor_vec','blocks_subject_vec','follows_actor_vec','follows_subject_vec']
for c in vec_cols:
    if c not in user_df.columns:
        user_df[c] = [[0]*7 for _ in range(len(user_df))]
    else:
        user_df[c] = user_df[c].apply(ensure_vec7)

df = user_df.copy()

Loaded user activity rows: 48625


In [4]:
# Basic statistical features
# Posts features (week 1: days 0..6)
df['posts_total'] = df['posts_vec'].apply(sum)
df['posts_avg'] = df['posts_vec'].apply(lambda v: sum(v)/7.0)
df['posts_std'] = df['posts_vec'].apply(lambda v: float(np.std(v)))
df['posts_active_days'] = df['posts_vec'].apply(lambda v: sum(1 for x in v if x>0))
df['posts_day0'] = df['posts_vec'].apply(lambda v: int(v[0]))

# Blocks features (initiated vs received)
df['blocks_initiated_total'] = df['blocks_actor_vec'].apply(sum)
df['blocks_received_total'] = df['blocks_subject_vec'].apply(sum)
df['blocks_initiated_active_days'] = df['blocks_actor_vec'].apply(lambda v: sum(1 for x in v if x>0))
df['blocks_received_active_days'] = df['blocks_subject_vec'].apply(lambda v: sum(1 for x in v if x>0))

# Follows features (made vs received)
df['follows_made_total'] = df['follows_actor_vec'].apply(sum)
df['follows_received_total'] = df['follows_subject_vec'].apply(sum)
df['follows_made_active_days'] = df['follows_actor_vec'].apply(lambda v: sum(1 for x in v if x>0))
df['follows_received_active_days'] = df['follows_subject_vec'].apply(lambda v: sum(1 for x in v if x>0))

In [5]:
# Temporal / recency features: first/last active day within week 1 (0..6), -1 if none

def first_active_day(vec):
    for i, val in enumerate(vec):
        if val and val > 0:
            return i
    return -1

def last_active_day(vec):
    for i in range(len(vec)-1, -1, -1):
        if vec[i] and vec[i] > 0:
            return i
    return -1

# Posts recency
df['posts_first_active_day'] = df['posts_vec'].apply(first_active_day)
df['posts_last_active_day'] = df['posts_vec'].apply(last_active_day)

# Follows recency (made vs received)
df['follows_made_first_day'] = df['follows_actor_vec'].apply(first_active_day)
df['follows_made_last_day'] = df['follows_actor_vec'].apply(last_active_day)
# df['follows_received_first_day'] = df['follows_subject_vec'].apply(first_active_day)
# df['follows_received_last_day'] = df['follows_subject_vec'].apply(last_active_day)

# Blocks recency (initiated vs received)
df['blocks_initiated_first_day'] = df['blocks_actor_vec'].apply(first_active_day)
df['blocks_initiated_last_day'] = df['blocks_actor_vec'].apply(last_active_day)
# df['blocks_received_first_day'] = df['blocks_subject_vec'].apply(first_active_day)
# df['blocks_received_last_day'] = df['blocks_subject_vec'].apply(last_active_day)

# Aggregate recency: most recent activity day across types (max of last_day values)
df['last_active_overall'] = df[['posts_last_active_day', 'follows_made_last_day', 'blocks_initiated_last_day']].max(axis=1)

# Feature Selection

In [6]:
# Select final feature columns computed from user activity vectors
feature_columns = [
    # Basic statistics
    'posts_total', 'posts_avg', 'posts_std', 'posts_active_days', 'posts_day0',

    'blocks_initiated_total', 'blocks_received_total', 'blocks_initiated_active_days', 'blocks_received_active_days',

    'follows_made_total', 'follows_received_total', 'follows_made_active_days', 'follows_received_active_days',
    
    # Recency features
    'posts_first_active_day', 'posts_last_active_day',
    'follows_made_first_day', 'follows_made_last_day',
    'blocks_initiated_first_day', 'blocks_initiated_last_day',
    'last_active_overall',
    
]

# Ensure features exist in df (fill missing with 0)
for c in feature_columns:
    if c not in df.columns:
        df[c] = 0

# Create final feature dataset
X = df[feature_columns].fillna(0)
y = target_table['posts_week2']

print(f"Final feature set shape: {X.shape}")
print(f"Feature columns: {list(X.columns)}")

# Check correlation with target
correlations = X.corrwith(y).sort_values(ascending=False)
print("\nTop features correlated with target:")
print(correlations.head(10))

Final feature set shape: (48625, 20)
Feature columns: ['posts_total', 'posts_avg', 'posts_std', 'posts_active_days', 'posts_day0', 'blocks_initiated_total', 'blocks_received_total', 'blocks_initiated_active_days', 'blocks_received_active_days', 'follows_made_total', 'follows_received_total', 'follows_made_active_days', 'follows_received_active_days', 'posts_first_active_day', 'posts_last_active_day', 'follows_made_first_day', 'follows_made_last_day', 'blocks_initiated_first_day', 'blocks_initiated_last_day', 'last_active_overall']

Top features correlated with target:
posts_total                     0.671576
posts_avg                       0.671576
posts_std                       0.525834
posts_active_days               0.336448
posts_last_active_day           0.197260
follows_made_active_days        0.175206
posts_day0                      0.170494
blocks_initiated_active_days    0.164979
blocks_received_active_days     0.164236
blocks_initiated_last_day       0.158515
dtype: float64


# Save the file 

In [8]:
# Save features and target separately
features_output_path = "../data/posting/featured/features.parquet"
target_output_path = "../data/posting/featured/target.parquet"

# Save as separate files - Convert y to DataFrame first
X.to_parquet(features_output_path, index=False)
y.to_frame().to_parquet(target_output_path, index=False)  # Convert Series to DataFrame

print(f"\nFeatures saved to: {features_output_path}")
print(f"Target saved to: {target_output_path}")


Features saved to: ../data/posting/featured/features.parquet
Target saved to: ../data/posting/featured/target.parquet
